# A2.3 · Delegation that narrows, and survives audit

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.2 · Bootstrapping the first credential](https://spbreed.github.io/cyber-commons/lessons/A2.2.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Delegation that widens is not delegation, it is escalation with paperwork. The rule is simple to state and almost never enforced: what is issued must be a subset of what was presented, and inside the recipient's own ceiling.

## 2 · The framework

```
   presented                issued
   {repo:read, repo:write}  ---> {repo:read}        OK   subset
   {repo:read}              ---> {repo:read, write} NO   widening
   {repo:write}             ---> {repo:write}       NO   above the
                                 to triage-agent         recipient's ceiling

   two rules, checked at every hop, and the chain recorded for the audit
```

**Mitigates: T3 Privilege Compromise · T8 Repudiation · T14 Human Attacks.**

The agent has its own identity now. This lesson is about carrying the user's
authority alongside it without either losing it or amplifying it.

Delegation — RFC 8693 token exchange, "on behalf of" — issues a new token for a
downstream hop. It must satisfy **two** narrowing rules, and checking only one
is the common and dangerous mistake:

**Subset of presented.** The issued token carries no more scope than the
incoming one. Stops the agent inventing authority.

**Within the actor's ceiling.** The issued token carries no more than the
receiving agent is ever permitted to hold. Stops a privileged user handing an
agent authority the agent must never have.

Neither is sufficient alone, and they fail in opposite directions. Subset-only
lets an admin's request give a low-trust agent `db:admin` — legitimately, and it
looks correct in every log. Ceiling-only lets an agent exceed the person who
asked, which is A1.6.

The issued scope is the **intersection**.

The second half is the **actor chain**: `dana → orchestrator → patch-agent`,
recorded on the token and carried to every hop. That is what makes A1.13
answerable and what makes A1.16's laundering path visible, because the
composition is written down rather than inferred.

> **What this control closes.**
>
> Both rules, every hop. Checking one is how a privileged user hands an agent authority it must never hold — legitimately, and invisibly.

## 3 · The control

In [ ]:
CEILINGS = {
 "dana@corp":     {"reports:read", "reports:write"},
 "priya@corp":    {"reports:read", "reports:write", "db:admin"},
 "orchestrator":  {"reports:read", "reports:write"},
 "patch-agent":   {"reports:read"},
}

class DelegationError(Exception): pass

def exchange(presented_scopes, presented_chain, actor, requested):
    """One hop. BOTH narrowing rules, then record the chain."""
    requested = set(requested)
    if not requested <= set(presented_scopes):                    # rule 1
        raise DelegationError(
            f"widening: {sorted(requested - set(presented_scopes))} not in presented")
    ceiling = CEILINGS[actor]
    issued = requested & ceiling                                  # rule 2
    if issued != requested:
        print(f"      (narrowed by {actor} ceiling: dropped "
              f"{sorted(requested - ceiling)})")
    return {"scopes": issued, "chain": presented_chain + [actor]}

# --- the honest path -------------------------------------------------------
user = {"scopes": CEILINGS["dana@corp"], "chain": ["dana@corp"]}
hop1 = exchange(user["scopes"], user["chain"], "orchestrator",
                {"reports:read", "reports:write"})
hop2 = exchange(hop1["scopes"], hop1["chain"], "patch-agent", {"reports:read"})
print(f"   chain  : {' -> '.join(hop2['chain'])}")
print(f"   scopes : {sorted(hop2['scopes'])}")

# --- a privileged user, and the rule that saves you ------------------------
print("\npriya holds db:admin. She asks the same low-trust agent to use it:")
priv = {"scopes": CEILINGS["priya@corp"], "chain": ["priya@corp"]}
subset_only = {"db:admin"} <= priv["scopes"]
issued = exchange(priv["scopes"], priv["chain"], "patch-agent", {"db:admin"})
print(f"   subset-of-presented alone would allow it : {subset_only}")
print(f"   scopes actually issued                   : {sorted(issued['scopes']) or 'none'}")
print()
print("Subset-only says yes - she really does hold db:admin. The ceiling rule")
print("issues the intersection, which is empty, because patch-agent may never")
print("hold it no matter who asks.")
assert subset_only and not issued["scopes"]
assert hop2["chain"] == ["dana@corp", "orchestrator", "patch-agent"]

## What you just proved

A two-hop delegation narrows to `reports:read` and records the chain `dana → orchestrator → patch-agent`. A privileged user's request for `db:admin` passes subset-of-presented and still issues nothing, because the receiving agent's ceiling is empty of it.

## Your turn

Find your token exchange and check which of the two rules it implements. Most implement subset-of-presented, because it is the one the specification example shows.

---

**Next → [A2.4 · Just-in-time authority](https://spbreed.github.io/cyber-commons/lessons/A2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*